# Sketch2Shape: Kaggle trial run

Checks, before the event, that the planned models run on Kaggle's free **GPU T4 x2**, and measures time and GPU memory.

**Settings** (right panel, Session options): Accelerator **GPU T4 x2**, Internet **On**.

**Run:** Run All. The first run takes roughly 30-60 minutes (installs, downloads, compiling). Keep the tab open.

**When it finishes:** copy the block printed under `REPORT` and paste it back to Claude. Also say which generated images look most like the sketch, and whether the share link at the bottom opened on your laptop.

To test your own sketch, attach it as input data (Add Input) and change `SUBJECT` in the sketch cell. Otherwise a built-in mug drawing is used.

This notebook only tests the environment. It is not project code; don't submit it.

In [ ]:
# 1. Environment check
import os, sys, json, time, glob, shutil, platform, subprocess

OUT = "/kaggle/working/trial"
os.makedirs(OUT, exist_ok=True)
os.environ["HF_HOME"] = "/tmp/hf"                   # model cache stays out of the saved output
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"          # T4 only, so the CUDA build is quicker
REPO = "/tmp/Hunyuan3D-2"
REPORT_PATH = f"{OUT}/report.json"
report = {"started": time.strftime("%Y-%m-%d %H:%M:%S")}


def save_report():
    with open(REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)


def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()


import torch
report["env"] = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "gpus": sh("nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader"),
    "nvcc": sh("nvcc --version | tail -n 2"),
    "tmp_free_gb": round(shutil.disk_usage("/tmp").free / 1e9, 1),
    "run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "unknown"),
}
save_report()
print(json.dumps(report["env"], indent=2))
assert torch.cuda.device_count() >= 2, "Choose 'GPU T4 x2' in Session options, then restart the session."

In [ ]:
# 2. Log both GPUs' memory every 0.5 s for the whole run, to find the true peaks
gpu_log = open(f"{OUT}/gpu_log.csv", "w")
logger = subprocess.Popen(
    ["nvidia-smi", "--query-gpu=timestamp,index,memory.used", "--format=csv,noheader,nounits", "-lms", "500"],
    stdout=gpu_log, stderr=subprocess.DEVNULL)
print("GPU logger running")

In [ ]:
# 3. Install Hunyuan3D-2 and its requirements
t0 = time.time()
if not os.path.exists(REPO):
    print(sh(f"git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2 {REPO}")[-500:])
print(sh(f"cd {REPO} && pip install -q -r requirements.txt && pip install -q -e .")[-3000:])
report["install_s"] = round(time.time() - t0)
report["pip_check"] = sh("pip check")[-1500:]

# Newer diffusers refuses to run a local custom pipeline file unless trust_remote_code=True.
# The file is Hunyuan's own hunyuanpaint/pipeline.py from the repo cloned above.
mv_utils = f"{REPO}/hy3dgen/texgen/utils/multiview_utils.py"
s = open(mv_utils).read()
old = "custom_pipeline=custom_pipeline_path, torch_dtype=torch.float16)"
if "trust_remote_code" not in s and old in s:
    open(mv_utils, "w").write(s.replace(old, old[:-1] + ", trust_remote_code=True)"))
report["paint_patch"] = "trust_remote_code=True" in open(mv_utils).read()
print("texture pipeline patch applied:", report["paint_patch"])
save_report()
print("install took", report["install_s"], "s")
print("pip check:\n", report["pip_check"])

In [ ]:
# 4. Compile the two CUDA extensions the texture stage needs, as wheels
#    Saved with a notebook version, these wheels let later sessions skip compiling.
WHEELS = "/kaggle/working/wheels"
os.makedirs(WHEELS, exist_ok=True)
t0 = time.time()
report["build"] = {}
for ext in ["custom_rasterizer", "differentiable_renderer"]:
    src = f"{REPO}/hy3dgen/texgen/{ext}"
    before = set(os.listdir(WHEELS))
    log = sh(f"cd {src} && pip wheel . --no-deps --no-build-isolation -w {WHEELS}")
    if set(os.listdir(WHEELS)) == before:
        log += "\n[wheel build failed, trying setup.py install]\n" + sh(f"cd {src} && python setup.py install")
        report["build"][ext] = "wheel failed: " + log[-1200:]
    else:
        report["build"][ext] = "wheel ok"
    print(f"--- {ext} ---\n{log[-1500:]}\n")
if os.listdir(WHEELS):
    print(sh(f"pip install -q --force-reinstall --no-deps {WHEELS}/*.whl")[-1000:])
report["build_s"] = round(time.time() - t0)
check = sh(f'cd {REPO} && python -c "from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline; '
           f'from hy3dgen.texgen import Hunyuan3DPaintPipeline; print(\'IMPORT_OK\')"')
report["texture_import_ok"] = "IMPORT_OK" in check
if not report["texture_import_ok"]:
    report["texture_import_error"] = check[-1500:]
save_report()
print("build took", report["build_s"], "s; wheels:", os.listdir(WHEELS))
print("texture stage imports:", "OK" if report["texture_import_ok"] else "FAILED\n" + check[-1500:])

In [ ]:
# 5. Test sketch: your own attached image, or a built-in mug drawing
SUBJECT = "a ceramic coffee mug"   # change this to describe your own sketch

import numpy as np, cv2
from PIL import Image


def find_user_sketch():
    for ext in ("png", "jpg", "jpeg", "webp"):
        hits = sorted(glob.glob(f"/kaggle/input/**/*.{ext}", recursive=True))
        if hits:
            return hits[0]
    return None


def draw_mug(size=1024):
    img = np.full((size, size), 255, np.uint8)
    t = 7
    cv2.ellipse(img, (480, 300), (210, 60), 0, 0, 360, 0, t)      # rim
    cv2.line(img, (270, 300), (290, 780), 0, t)                    # left side
    cv2.line(img, (690, 300), (670, 780), 0, t)                    # right side
    cv2.ellipse(img, (480, 780), (190, 55), 0, 0, 180, 0, t)       # base
    cv2.ellipse(img, (680, 540), (120, 150), 0, -90, 90, 0, t)     # handle, outer
    cv2.ellipse(img, (680, 540), (65, 90), 0, -90, 90, 0, t)       # handle, inner
    return img


src = find_user_sketch()
if src:
    g = cv2.cvtColor(cv2.imread(src), cv2.COLOR_BGR2GRAY)
    g = cv2.adaptiveThreshold(cv2.GaussianBlur(g, (5, 5), 0), 255,
                              cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 10)
    h, w = g.shape
    s = max(h, w)
    canvas = np.full((s, s), 255, np.uint8)
    canvas[(s - h) // 2:(s - h) // 2 + h, (s - w) // 2:(s - w) // 2 + w] = g
    g = cv2.resize(canvas, (1024, 1024), interpolation=cv2.INTER_AREA)
    report["sketch"] = "user sketch: " + src
else:
    g = draw_mug()
    report["sketch"] = "built-in mug drawing"
report["subject"] = SUBJECT

Image.fromarray(g).convert("RGB").save(f"{OUT}/sketch_bw.png")        # black lines on white
Image.fromarray(255 - g).convert("RGB").save(f"{OUT}/sketch_wb.png")  # white lines on black
save_report()
display(Image.fromarray(np.hstack([g, 255 - g])).resize((768, 384)))

In [ ]:
%%writefile /kaggle/working/trial/sdxl_trial.py
# SDXL + xinsir scribble ControlNet on one T4: time and memory for both line polarities.
import json, sys, time, torch
from PIL import Image
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, AutoencoderKL

out_dir, subject = sys.argv[1], sys.argv[2]
res = {}
t0 = time.time()
controlnet = ControlNetModel.from_pretrained("xinsir/controlnet-scribble-sdxl-1.0", torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", controlnet=controlnet, vae=vae,
    torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
if hasattr(pipe.vae, "enable_slicing"):   # newer diffusers removed pipe.enable_vae_slicing()
    pipe.vae.enable_slicing()
pipe.set_progress_bar_config(disable=True)
res["load_s"] = round(time.time() - t0, 1)

prompt = (f"{subject}, single object, centered, whole object in frame, product photo, "
          "plain white background, soft even studio lighting, high detail")
negative = ("multiple objects, cropped, text, watermark, cast shadow, ground, scene, "
            "cluttered background, blurry, sketch, line art, drawing")


def run(control_path, n):
    control = Image.open(control_path).convert("RGB")
    torch.cuda.reset_peak_memory_stats()
    t = time.time()
    images = pipe(prompt, negative_prompt=negative, image=control,
                  controlnet_conditioning_scale=0.7, num_inference_steps=25, guidance_scale=6.0,
                  num_images_per_prompt=n, width=1024, height=1024,
                  generator=torch.Generator("cuda").manual_seed(42)).images
    return images, round(time.time() - t, 1), round(torch.cuda.max_memory_reserved() / 2**30, 2)


for pol in ("bw", "wb"):
    n = 2
    try:
        images, secs, peak = run(f"{out_dir}/sketch_{pol}.png", n)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        n = 1
        images, secs, peak = run(f"{out_dir}/sketch_{pol}.png", n)
    for i, im in enumerate(images):
        im.save(f"{out_dir}/sdxl_{pol}_{i}.png")
    res[pol] = {"images": n, "seconds": secs, "peak_gb": peak}
    print(pol, res[pol], flush=True)

json.dump(res, open(f"{out_dir}/sdxl_result.json", "w"), indent=2)

In [ ]:
# 6. Run the SDXL test on GPU 0
t0 = time.time()
!CUDA_VISIBLE_DEVICES=0 python {OUT}/sdxl_trial.py {OUT} "{SUBJECT}" 2>&1 | tail -n 40
report["sdxl_wall_s"] = round(time.time() - t0)
try:
    report["sdxl"] = json.load(open(f"{OUT}/sdxl_result.json"))
except FileNotFoundError:
    report["sdxl"] = "FAILED: see the log above"
save_report()

imgs = sorted(glob.glob(f"{OUT}/sdxl_*.png"))
if imgs:
    tiles = [Image.open(p).resize((320, 320)) for p in imgs]
    grid = Image.new("RGB", (320 * len(tiles), 320), "white")
    for i, tile in enumerate(tiles):
        grid.paste(tile, (320 * i, 0))
    display(grid)
    print("left to right:", [os.path.basename(p) for p in imgs])
    print("bw = control image with black lines on white, wb = white lines on black")

In [ ]:
%%writefile /kaggle/working/trial/hy3d_trial.py
# Hunyuan3D-2 on one T4: shape first, then texture after the shape model is unloaded.
import gc, json, sys, time, traceback, torch
from PIL import Image
from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen import (Hunyuan3DDiTFlowMatchingPipeline, FloaterRemover,
                              DegenerateFaceRemover, FaceReducer)

out_dir, jobs_path = sys.argv[1], sys.argv[2]
jobs = json.load(open(jobs_path))
res = {"jobs": {}}


def peak_gb():
    return round(torch.cuda.max_memory_reserved() / 2**30, 2)


rembg = BackgroundRemover()
images = {}
for job in jobs:
    im = Image.open(job["image"])
    im = im.convert("RGBA") if im.mode == "RGBA" else rembg(im.convert("RGB"))
    im.save(f"{out_dir}/prepared_{job['name']}.png")
    images[job["name"]] = im

t0 = time.time()
try:
    shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
        "tencent/Hunyuan3D-2", subfolder="hunyuan3d-dit-v2-0-turbo", use_safetensors=True)
    shape.enable_flashvdm()
    steps, res["shape_model"] = 5, "dit-v2-0-turbo + flashvdm"
except Exception as e:
    print("turbo model failed to load, using the base model:", repr(e), flush=True)
    shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("tencent/Hunyuan3D-2")
    steps, res["shape_model"] = 50, "dit-v2-0"
res["shape_load_s"] = round(time.time() - t0, 1)

meshes = {}
cleanup = [FloaterRemover(), DegenerateFaceRemover(), FaceReducer()]
for job in jobs:
    r = res["jobs"].setdefault(job["name"], {})
    try:
        torch.cuda.reset_peak_memory_stats()
        t = time.time()
        mesh = shape(image=images[job["name"]], num_inference_steps=steps,
                     generator=torch.manual_seed(1234))[0]
        r["shape_s"], r["shape_peak_gb"] = round(time.time() - t, 1), peak_gb()
        for step in cleanup:
            mesh = step(mesh)
        r["faces"] = int(len(mesh.faces))
        mesh.export(f"{out_dir}/{job['name']}_shape.glb")
        meshes[job["name"]] = mesh
    except Exception:
        r["shape_error"] = traceback.format_exc()[-800:]
    print(job["name"], r, flush=True)

del shape
gc.collect()
torch.cuda.empty_cache()

from hy3dgen.texgen import Hunyuan3DPaintPipeline
t0 = time.time()
paint = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2")
res["paint_load_s"] = round(time.time() - t0, 1)
res["paint_cpu_offload"] = False
for job in jobs:
    if not job["texture"] or job["name"] not in meshes:
        continue
    r = res["jobs"][job["name"]]
    for attempt in ("gpu", "cpu_offload"):
        try:
            torch.cuda.reset_peak_memory_stats()
            t = time.time()
            textured = paint(meshes[job["name"]], image=images[job["name"]])
            r["paint_s"], r["paint_peak_gb"], r["paint_mode"] = round(time.time() - t, 1), peak_gb(), attempt
            textured.export(f"{out_dir}/{job['name']}_textured.glb")
            break
        except torch.cuda.OutOfMemoryError:
            print(job["name"], "texture ran out of GPU memory", flush=True)
            torch.cuda.empty_cache()
            if not res["paint_cpu_offload"]:
                paint.enable_model_cpu_offload()
                res["paint_cpu_offload"] = True
        except Exception:
            r["paint_error"] = traceback.format_exc()[-800:]
            break
    else:
        r["paint_error"] = "out of GPU memory, even with CPU offload"
    print(job["name"], r, flush=True)

json.dump(res, open(f"{out_dir}/hy3d_result.json", "w"), indent=2)

In [ ]:
# 7. Run Hunyuan3D-2 on GPU 1: its demo image, one SDXL image per polarity, and the raw sketch (shape only)
jobs = [
    {"name": "demo", "image": f"{REPO}/assets/demo.png", "texture": True},
    {"name": "sdxl_bw", "image": f"{OUT}/sdxl_bw_0.png", "texture": True},
    {"name": "sdxl_wb", "image": f"{OUT}/sdxl_wb_0.png", "texture": True},
    {"name": "raw_sketch", "image": f"{OUT}/sketch_bw.png", "texture": False},
]
jobs = [j for j in jobs if os.path.exists(j["image"])]
json.dump(jobs, open(f"{OUT}/jobs.json", "w"))
t0 = time.time()
!cd {REPO} && CUDA_VISIBLE_DEVICES=1 python {OUT}/hy3d_trial.py {OUT} {OUT}/jobs.json 2>&1 | tail -n 60
report["hy3d_wall_s"] = round(time.time() - t0)
try:
    report["hy3d"] = json.load(open(f"{OUT}/hy3d_result.json"))
except FileNotFoundError:
    report["hy3d"] = "FAILED: see the log above"
save_report()

In [ ]:
# 8. Check every exported GLB: does it load, is it non-empty, is it flat, does it carry a texture?
import trimesh

checks = {}
for path in sorted(glob.glob(f"{OUT}/*.glb")):
    try:
        scene = trimesh.load(path, force="scene")
        geoms = list(scene.geometry.values())
        ext = scene.extents
        textured = False
        for geom in geoms:
            mat = getattr(geom.visual, "material", None)
            if getattr(mat, "baseColorTexture", None) is not None or getattr(mat, "image", None) is not None:
                textured = True
        checks[os.path.basename(path)] = {
            "vertices": int(sum(len(geom.vertices) for geom in geoms)),
            "faces": int(sum(len(getattr(geom, "faces", [])) for geom in geoms)),
            "flatness": round(float(min(ext) / max(ext)), 3) if ext is not None and max(ext) > 0 else 0.0,
            "textured": textured,
            "size_mb": round(os.path.getsize(path) / 1e6, 2),
        }
    except Exception as e:
        checks[os.path.basename(path)] = {"error": repr(e)}
report["glb_checks"] = checks
save_report()
for name, c in checks.items():
    print(name, c)

In [ ]:
# 9. Stop the GPU logger and record peak memory per GPU (MiB, as nvidia-smi saw it)
logger.terminate()
gpu_log.close()
peaks = {}
for line in open(f"{OUT}/gpu_log.csv"):
    parts = [p.strip() for p in line.split(",")]
    if len(parts) == 3 and parts[2].isdigit():
        peaks["gpu" + parts[1]] = max(peaks.get("gpu" + parts[1], 0), int(parts[2]))
report["nvidia_smi_peak_mib"] = peaks
report["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
save_report()
print("===== REPORT: copy everything below and paste it to Claude =====")
print(json.dumps(report, indent=1)[:15000])

In [ ]:
# 10. Share-link test (interactive sessions only): open the printed gradio.live link on your laptop
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Interactive":
    import gradio as gr
    glbs = sorted(glob.glob(f"{OUT}/*_textured.glb")) or sorted(glob.glob(f"{OUT}/*.glb"))
    with gr.Blocks() as demo:
        gr.Markdown("Sketch2Shape trial. If this page opens on your laptop, the share link works.")
        for p in glbs[:3]:
            gr.Model3D(value=p, label=os.path.basename(p))
    demo.launch(share=True)
else:
    print("Skipped in a saved (batch) run.")

## Optional: keep the compiled wheels for later sessions

Only if the run above worked. Click **Save Version**, then **Save & Run All**. This repeats the run in the background, using about the same GPU time again. When that version finishes, its output holds `wheels/`. Make a private dataset from that output (or download the `wheels` folder and upload it as a new dataset). Later notebooks attach it with Add Input and install with `pip install /kaggle/input/<dataset>/wheels/*.whl`, which skips compiling.